# KAN-MCP 最小复现（Colab 默认环境）

在 Colab 中选择 GPU 运行时，按顺序执行。主要操作使用 `!` 命令，指标解析与写入 Google Drive 使用 Python。默认各运行 MOSI、MOSEI 一次；论文表 1 为五次均值。不安装仓库的依赖；直接使用 Colab 默认环境。如果默认环境缺少包，训练日志会保留报错。

论文：[KAN-MCP.pdf](https://drive.google.com/file/d/1qPrmCZlPIvZ-cQ0NPVu1voXqFkgmnWxL/view)；代码：[Scream9371/KAN-MCP](https://github.com/Scream9371/KAN-MCP)。

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, sys
from pexpect.exceptions import TIMEOUT

try:
    drive.mount('/content/drive', timeout_ms=6000)
except TIMEOUT as exc:
    raise RuntimeError('Google Drive 挂载超时。请在 Colab 网页端打开 notebook，完成弹出的账号授权；若仍失败，重启运行时后重试此格。') from exc
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_dir = Path('/content/drive/MyDrive/KAN-MCP-reproduction') / run_id
out_dir.mkdir(parents=True, exist_ok=False)
repo_dir = Path('/content') / f'KAN-MCP-{run_id}'
print('结果目录:', out_dir)
!nvidia-smi

In [ ]:
if repo_dir.exists():
    !git -C {repo_dir} pull --ff-only
else:
    !git clone https://github.com/Scream9371/KAN-MCP.git {repo_dir}


## 数据

仓库不包含数据文件。从 README 中的 Dropbox 链接下载到运行时。若数据已在 Google Drive，可改成 `!cp` 从 Drive 复制到 `{repo_dir}/datasets/`。

In [ ]:
data_dir = repo_dir / 'datasets'
data_dir.mkdir(exist_ok=True)
mosi = data_dir / 'mosi.pkl'
mosei = data_dir / 'mosei.pkl'
if not mosi.exists():
    !curl -fL --retry 3 'https://www.dropbox.com/s/sv94igp7zi3rsj1/mosi.pkl?dl=1' -o {mosi}
if not mosei.exists():
    !curl -fL --retry 3 'https://www.dropbox.com/s/995dvj3506gvk5a/mosei.pkl?dl=1' -o {mosei}
for target in (mosi, mosei):
    assert target.exists() and target.stat().st_size > 1_000_000, f'数据下载失败或文件异常: {target}'
    print(target.name, f'{target.stat().st_size/1024**2:.1f} MiB')


## 训练与持久化

从 Fork 直接运行训练脚本。`bash -o pipefail` 确保训练失败时即使经过 `tee` 仍能检测到错误。输出实时写入 Google Drive。脚本最后的 `====RESULT====` 是验证损失最低 epoch 对应的测试指标。

In [ ]:
import os, subprocess
assert (repo_dir / 'train_mosi.py').exists(), '仓库尚未克隆成功'
assert 'CUDA_VISIBLE_DEVICES' not in (repo_dir / 'train_mosi.py').read_text()
assert 'CUDA_VISIBLE_DEVICES' not in (repo_dir / 'train_mosei.py').read_text()
!python -c 'import sys, torch, transformers; print(sys.version.split()[0], torch.__version__, torch.version.cuda, transformers.__version__); assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))'
commit = subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip()
manifest = {'repo_url': 'https://github.com/Scream9371/KAN-MCP.git', 'commit': commit,
            'python': sys.version, 'seeds': {'mosi': 1024, 'mosei': 1025}}
(out_dir / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
os.chdir(repo_dir)
print('Commit:', commit, '运行目录:', Path.cwd())


In [ ]:
mosi_log = out_dir / 'mosi_seed1024.log'
!bash -o pipefail -c 'python train_mosi.py --seed 1024 2>&1 | tee "{mosi_log}"'
assert mosi_log.exists() and '====RESULT====' in mosi_log.read_text(encoding='utf-8'), f'MOSI 训练未完成，查看 {mosi_log}'


In [ ]:
mosei_log = out_dir / 'mosei_seed1025.log'
!bash -o pipefail -c 'python train_mosei.py --seed 1025 2>&1 | tee "{mosei_log}"'
assert mosei_log.exists() and '====RESULT====' in mosei_log.read_text(encoding='utf-8'), f'MOSEI 训练未完成，查看 {mosei_log}'


In [ ]:
import re
pattern = re.compile(r'acc7:([^,]+), acc2:([^,]+), f1:([^,]+), mae:([^,]+), corr:([^\s]+)')
results = []
for name, seed, log_path in [('mosi', 1024, mosi_log), ('mosei', 1025, mosei_log)]:
    log = log_path.read_text(encoding='utf-8')
    assert '====RESULT====' in log, f'{log_path} 中没有最终结果'
    match = pattern.search(log.rsplit('====RESULT====', 1)[1])
    assert match, f'无法解析 {log_path} 中的结果'
    row = {'dataset': name, 'seed': seed, **dict(zip(['acc7','acc2','f1','mae','corr'], map(float, match.groups())))}
    results.append(row)
(out_dir / 'results.json').write_text(json.dumps(results, ensure_ascii=False, indent=2))
for row in results: print(row)
print('论文表 1: MOSI Acc7 48.3%, Acc2 89.4%, F1 89.4%, MAE 0.615, Corr 0.857；MOSEI 53.9%, 87.7%, 87.6%, 0.522, 0.788。')
print('本次 Acc/F1 输出为 0–1 比例；论文以百分数呈现。结果已保存:', out_dir)